# Notebook 3: End-to-End Demo — PronounceAI Module 1

**Speech-to-Text with Fine-Tuned Whisper**

This demo notebook lets you:
1. Provide an audio file path (or record from microphone)
2. Run transcription using the fine-tuned Whisper model
3. See the predicted text output immediately

> **Prerequisite:** Run Notebook 1 first to generate the saved model weights.

## Step 1: Install Dependencies

In [1]:
%pip install transformers torch torchaudio soundfile librosa ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Imports

In [2]:
import torch
import torchaudio
import numpy as np
import soundfile as sf
import librosa
from pathlib import Path
from transformers import WhisperProcessor, WhisperForConditionalGeneration

WEIGHTS_DIR = Path("../weights/whisper-base-librispeech")
SAMPLE_RATE = 16_000
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


## Step 3: Load the Fine-Tuned Model

In [3]:
print(f"Loading model from: {WEIGHTS_DIR.resolve()}")
processor = WhisperProcessor.from_pretrained(str(WEIGHTS_DIR))
model     = WhisperForConditionalGeneration.from_pretrained(str(WEIGHTS_DIR)).to(device)
model.eval()
print("Model ready!")

Loading model from: E:\UMD\Data_641_PCS1\NLP_project\PronounceAI\model\module_1\weights\whisper-base-librispeech


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Model ready!


## Step 4: Transcription Helper Function

In [4]:
def load_audio(file_path: str) -> np.ndarray:
    """
    Load any audio file and resample to 16 kHz mono.
    Supports: .flac, .wav, .mp3, .ogg, etc.
    """
    waveform, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)
    return waveform


def transcribe(file_path: str) -> str:
    """Load an audio file and return its transcription."""
    print(f"Loading audio: {file_path}")
    audio_array = load_audio(file_path)
    duration    = len(audio_array) / SAMPLE_RATE
    print(f"Duration: {duration:.1f} seconds")

    inputs = processor(
        audio_array,
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = model.generate(inputs)

    transcript = processor.decode(predicted_ids[0], skip_special_tokens=True)
    return transcript

## Step 5: Transcribe from File Path

Change `AUDIO_FILE` to the path of any audio file you want to transcribe.

In [5]:
# ── Edit this path to point to your audio file ─────────────────────────────
AUDIO_FILE = "../../../librispeech/LibriSpeech/test-clean/1089/134686/1089-134686-0000.flac"

if Path(AUDIO_FILE).exists():
    result = transcribe(AUDIO_FILE)
    print("\n" + "="*60)
    print("TRANSCRIPTION:")
    print(result)
    print("="*60)
else:
    print(f"File not found: {AUDIO_FILE}")
    print("Please update AUDIO_FILE to a valid path.")

Loading audio: ../../../librispeech/LibriSpeech/test-clean/1089/134686/1089-134686-0000.flac


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


Duration: 10.4 seconds


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transform


TRANSCRIPTION:
he hoped there would be stew for dinner turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick peppered flower fat and sauce


## Step 6: Interactive File Upload (Jupyter only)

Upload your own audio file and transcribe it on the fly.

In [6]:
import ipywidgets as widgets
from IPython.display import display, Audio as IPyAudio

uploader = widgets.FileUpload(accept="audio/*", multiple=False)
output   = widgets.Output()

def on_upload(change):
    with output:
        output.clear_output()
        if not uploader.value:
            return
        # Save uploaded bytes to a temp file
        uploaded_file = list(uploader.value.values())[0]
        tmp_path = Path("/tmp/uploaded_audio.wav")
        tmp_path.write_bytes(uploaded_file["content"])
        display(IPyAudio(str(tmp_path), autoplay=False))
        result = transcribe(str(tmp_path))
        print("\nTRANSCRIPTION:")
        print(result)

uploader.observe(on_upload, names="value")
display(widgets.VBox([uploader, output]))

## Step 7 (Optional): Microphone Input

Record directly from your microphone and transcribe.

> **Note:** Requires the `sounddevice` and `scipy` packages. This cell records 5 seconds of audio.

In [7]:
# ── Uncomment to enable microphone recording ────────────────────────────────
# !pip install sounddevice scipy

# import sounddevice as sd
# from scipy.io.wavfile import write as wav_write

# RECORD_SECONDS = 5
# MIC_FILE       = "/tmp/mic_recording.wav"

# print(f"Recording {RECORD_SECONDS} seconds... speak now!")
# recording = sd.rec(int(RECORD_SECONDS * SAMPLE_RATE), samplerate=SAMPLE_RATE,
#                    channels=1, dtype="float32")
# sd.wait()  # wait until recording is done
# wav_write(MIC_FILE, SAMPLE_RATE, recording)
# print("Recording done.")

# result = transcribe(MIC_FILE)
# print("\nTRANSCRIPTION:")
# print(result)

## Step 8: Batch Transcription of Multiple Files

In [8]:
def transcribe_folder(folder_path: str, extensions=(".flac", ".wav", ".mp3")) -> dict:
    """Transcribe all audio files in a folder. Returns {filename: transcript}."""
    folder = Path(folder_path)
    results = {}
    audio_files = [p for p in sorted(folder.rglob("*")) if p.suffix in extensions]
    print(f"Found {len(audio_files)} audio file(s) in {folder}")

    for audio_path in audio_files:
        transcript = transcribe(str(audio_path))
        results[audio_path.name] = transcript
        print(f"  {audio_path.name}: {transcript[:80]}...")

    return results


# Example usage (uncomment and set your folder):
# folder_results = transcribe_folder("/path/to/audio/folder")
# for fname, text in folder_results.items():
#     print(f"\n[{fname}]\n{text}")

---
## Module 1 Demo Complete

You have successfully run end-to-end speech-to-text transcription using PronounceAI Module 1.

**Next steps:**
- For detailed metrics, see **Notebook 2** (Evaluation)
- To re-train or fine-tune further, see **Notebook 1** (Training)